# Optimal Quoting under Adverse Selection and Price Reading
**Barzykin, Bergault, Gueant & Lemmel (2025)** | arXiv:2508.20225v3

---
This notebook reproduces the analytical derivations and numerical experiments of the paper.

> [WARNING] **Deviation - simulation scale**: The paper uses 1e5 paths. This notebook uses
> **5 000 paths** and **dt=2 s** to keep runtime manageable.


## 1. Setup
Imports, random seed, and global constants from Sec. 5.1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# --- Market parameters (Sec. 5.2) -------------------------------------------
SIZES      = np.array([1., 2., 5., 10., 20., 50.])  # Delta^k in M notional
K          = len(SIZES)

# Total baseline intensities Lambda^{1,k}_0 + Lambda^{2,k}_0 (day^-1)
LAMBDA0_TOTAL_DAY = np.array([2000., 800., 600., 400., 120., 80.])
LAMBDA0_TOTAL     = LAMBDA0_TOTAL_DAY / 86400.0   # convert to s^-1

KAPPA = 3.0                               # bp^-1, uniform across tiers/sizes
SIGMA = 100.0 / np.sqrt(86400.0)          # daily vol 100 bp -> bp/sqrt(s)
GAMMA = 1e-4                              # bp^-1 M^-1
E     = np.e                              # Euler's number in H^{n,k}(p) = L_0/k * e^{-1-kp}

print("Parameters loaded.")
print(f"  sigma={SIGMA:.4f} bp/sqrt(s), gamma={GAMMA:.1e} bp^-1 M^-1, kappa={KAPPA} bp^-1")
print(f"  Sizes (M): {SIZES}")
print(f"  Total Lambda_0 (day^-1): {LAMBDA0_TOTAL_DAY}")


## 2. Model and Perturbation Framework (Sec. 2-4)

### Reference price dynamics (Eq. 1)
$$dS_t = \sigma dB_t - \sum_{n,k}\tilde{\zeta}^{n,k}(\delta^{n,k,b}_t)dN^{n,k,b}_t
+ \sum_{n,k}\tilde{\zeta}^{n,k}(\delta^{n,k,a}_t)dN^{n,k,a}_t
+ \sum_n \tilde{J}^n\!\left(\sum_k w^{n,k}(\delta^{n,k,a}_t-\delta^{n,k,b}_t)\right)dt$$

- **Adverse selection** $\tilde{\zeta}^{n,k}=\varepsilon\zeta^{n,k}$: informed-flow price impact per trade
- **Price reading** $\tilde{J}^n=\varepsilon J^n$: skew-sniffer feedback via quote asymmetry

### First-order expansion (Sec. 3.3.3)
$$d^{n,k,b*}(q) = \delta^{n,k,b*}(q) + \varepsilon\,c^{n,k,b}\!\left[D^k_+f(q) + g^{n,k,b}_{\mathrm{PR}} + g^{n,k,b}_{\mathrm{AS}}\right] + o(\varepsilon)$$

Two components: **global** ($D^k_\pm f$, same for all tiers) + **tier-specific** ($g^{n,k,b/a}$).

### Quadratic approximation (Sec. 4)
With exponential functional forms $\Lambda^{n,k}(\delta)=\Lambda^{n,k}_0 e^{-\kappa\delta}$, $J^n(x)=x$,
$\zeta^{n,k}(\delta)=\alpha^{n,k}e^{\beta^{n,k}\delta}$, the baseline optimal quotes are:

$$\hat{\delta}^{n,k,b/a*}(q) = \frac{1}{\kappa} + \sigma\sqrt{\frac{\gamma e}{2\sum_{m,j}\Delta^j \Lambda^{m,j}_0\kappa^{m,j}}}\left(\pm q + \frac{\Delta^k}{2}\right)$$


## 3. Baseline Quotes and Helper Functions (Eq. 11, Sec. 5.1)

In [ ]:
# --- Exponential model helpers (Sec. 5.1) ------------------------------------
# H^{n,k}(p)    = Lambda_0/kappa * exp(-1 - kappa*p)
# H^{n,k''}(0)  = Lambda_0 * kappa * e^{-1}
# F^{2,1}_+     = 2 * sum_j Delta^j * H^{n,j''}(0)  = 2 * sum Delta*Lambda_0*kappa/e
# delta_tilde^{n,k*}(p) = p + 1/kappa  (Sec. 5.1)

def F21_plus(Lambda0):
    # Eq. (9) with H^{n,k''}(0) = Lambda_0 * kappa * e^-1
    return 2.0 * np.sum(SIZES * Lambda0 * KAPPA / E)

def weights_tier2():
    # w^{2,k} = exp(-sqrt(Delta^k)) / 1000  (Sec. 5.2)
    return np.exp(-np.sqrt(SIZES)) / 1000.0

def baseline_quotes(q, Lambda0):
    # Eq. (11): b_delta^{n,k,b/a*}(q) = 1/kappa + 2*A0 * (pm*q + Delta^k/2)
    # where A_0 = sigma * sqrt(gamma*e / (2 * sum Delta*Lambda_0*kappa)) / 2
    # (rho->0 limit of A_0 from Eq. (9))
    F = F21_plus(Lambda0)
    # 2*A_0 = sigma * sqrt(gamma / F) = sigma * sqrt(gamma*e/(2*sum(Delta*Lambda_0*kappa)))
    two_A0 = SIGMA * np.sqrt(GAMMA / F)
    q_ = np.asarray(q)
    if q_.ndim == 0:
        # scalar
        delta_b = 1.0/KAPPA + two_A0 * (float(q_) + SIZES/2.0)
        delta_a = 1.0/KAPPA + two_A0 * (-float(q_) + SIZES/2.0)
    else:
        # array of shape (n_sims,) -> output shape (K, n_sims)
        delta_b = 1.0/KAPPA + two_A0 * (q_[None,:] + SIZES[:,None]/2.0)
        delta_a = 1.0/KAPPA + two_A0 * (-q_[None,:] + SIZES[:,None]/2.0)
    return delta_b, delta_a

# Quick sanity check: myopic half-spread at q=0 for 1M size
db0, da0 = baseline_quotes(0.0, LAMBDA0_TOTAL)
print(f"Baseline half-spread at q=0:")
for k in range(K):
    print(f"  size {SIZES[k]:.0f}M: bid={db0[k]:.4f} bp, ask={da0[k]:.4f} bp")


## 4. Price Reading: Optimal Quote Adjustments (Figs 1 & 2)

**Setup** (Sec. 5.2): Two tiers; tier 2 contains skew sniffers with $w^{2,k}=e^{-\sqrt{\Delta^k}}/1000$,
tier 1 has $w^{1,k}=0$.

**Closed-form bid adjustment** (Sec. 5.2):
$$\frac{\hat{d}^{n,k,b*}(q)-\hat{\delta}^{n,k,b*}(q)}{\varepsilon}
= \underbrace{\frac{e\sum_{m,j}w^{m,j}}{\sum_{m,j}\Delta^j\Lambda_0^{m,j}\kappa^{m,j}}\left(q+\frac{\Delta^k}{2}\right)}_{\text{global}}
\underbrace{-\frac{q\,w^{n,k}}{\Delta^k\kappa\,\Lambda^{n,k}(\hat{\delta}^{n,k,b*}(q))}}_{\text{tier-specific}}$$


In [ ]:
# --- Price reading quote adjustment (Sec. 5.2, closed-form) ------------------
# Per the formula in Sec. 5.2 for exponential functional forms.
# c^{n,k} = 1 for exponential intensities.

def pr_adj_bid(q_arr, n_tier, Lambda0_1, Lambda0_2):
    Lambda0_tot = Lambda0_1 + Lambda0_2
    w2 = weights_tier2()            # shape (K,)

    # Denominator: sum_{m,j} Delta^j * Lambda_0^{m,j} * kappa  (uniform kappa)
    denom = KAPPA * np.sum(SIZES * Lambda0_tot)

    # Global component coefficient: e * sum(w) / denom
    sum_w = np.sum(w2)              # only tier 2 contributes
    global_coeff = E * sum_w / denom

    q = np.asarray(q_arr)
    # Baseline quotes for intensity evaluation: (K, Q)
    db_base, _ = baseline_quotes(q, Lambda0_tot)
    Lam_at_b = Lambda0_tot[:,None] * np.exp(-KAPPA * db_base)   # (K, Q)

    # Tier-specific weights
    w_n = w2 if n_tier == 1 else np.zeros(K)

    # Global term (bid, + sign per Sec. 5.2)
    global_bid = global_coeff * (q[None,:] + SIZES[:,None]/2.0)  # (K, Q)

    # Tier-specific term (bid): -q * w^{n,k} / (Delta^k * kappa * Lambda^{n,k}(delta))
    tier_bid = -(q[None,:] * w_n[:,None]) / (SIZES[:,None] * KAPPA * Lam_at_b)

    return global_bid + tier_bid    # (K, Q)


# --- Figure 1: SVS = 25% (Sec. 5.2) -----------------------------------------
q_grid = np.linspace(-60, 60, 300)
SVS = 0.25
L1, L2 = SVS * LAMBDA0_TOTAL, (1-SVS) * LAMBDA0_TOTAL

adj_t1 = pr_adj_bid(q_grid, 0, L1, L2)   # tier 1 (no sniffers)
adj_t2 = pr_adj_bid(q_grid, 1, L1, L2)   # tier 2 (skew sniffers)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(q_grid, adj_t1[0],  'k--',  lw=1.5, label='tier 1, 1M')
ax.plot(q_grid, adj_t1[3],  'k-',   lw=1.5, label='tier 1, 10M')
ax.plot(q_grid, adj_t2[0],  color='gray', lw=1.5, ls='--', label='tier 2, 1M')
ax.plot(q_grid, adj_t2[3],  color='gray', lw=1.5, ls='-',  label='tier 2, 10M')
ax.axhline(0, color='k', lw=0.5, ls=':'); ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set(xlim=(-60,60), ylim=(-1,1), xlabel='Inventory (M)', ylabel='Bid quote adjustment',
       title='Figure 1 - Price reading bid quote adjustment, SVS=25%')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('/tmp/fig1.png', dpi=110); plt.show()


**Figure 1**: At SVS=25%, many skew sniffers trade frequently so the global risk-aversion effect
dominates. Tier 2 quotes still show positive skew (more than the no-reading baseline),
because the sniffers contribute meaningfully to inventory risk reduction.


In [ ]:
# --- Figure 2: SVS = 75% (Sec. 5.2) -----------------------------------------
SVS = 0.75
L1, L2 = SVS * LAMBDA0_TOTAL, (1-SVS) * LAMBDA0_TOTAL

adj_t1 = pr_adj_bid(q_grid, 0, L1, L2)
adj_t2 = pr_adj_bid(q_grid, 1, L1, L2)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(q_grid, adj_t1[0],  'k--',  lw=1.5, label='tier 1, 1M')
ax.plot(q_grid, adj_t1[3],  'k-',   lw=1.5, label='tier 1, 10M')
ax.plot(q_grid, adj_t2[0],  color='gray', lw=1.5, ls='--', label='tier 2, 1M')
ax.plot(q_grid, adj_t2[3],  color='gray', lw=1.5, ls='-',  label='tier 2, 10M')
ax.axhline(0, color='k', lw=0.5, ls=':'); ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set(xlim=(-60,60), ylim=(-1,1), xlabel='Inventory (M)', ylabel='Bid quote adjustment',
       title='Figure 2 - Price reading bid quote adjustment, SVS=75%')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('/tmp/fig2.png', dpi=110); plt.show()


**Figure 2**: At SVS=75%, sniffers are rare and trade infrequently.
The de-skewing (tier-specific) effect dominates even for large sizes.
Tier 2 quotes show *less* skew than the no-reading baseline.


## 5. Monte Carlo Simulation Engine (Sec. 5.2 / 5.3)

Discrete-time simulation of the full model. At each step dt:
1. Brownian PnL contribution: $q\sigma\,dB$
2. Price reading drift on S (Eq. 1): $\varepsilon J^n(\text{skew})\,dt$
3. Poisson trade arrivals with intensities $\lambda^{n,k,b/a}=\Lambda^{n,k}(\delta^{n,k,b/a})$
4. Each trade updates inventory, cash, and reference price (adverse selection)

[WARNING] **Deviation**: Paper uses $T=10^4$ s, $10^5$ paths, continuous time.
This notebook uses **5 000 paths**, $dt=2$ s, $T=10^4$ s.


In [ ]:
# --- Simulation parameters --------------------------------------------------
N_SIMS  = 5_000
T_SIM   = 10_000.0   # seconds
DT      = 2.0        # time step (s)
N_STEPS = int(T_SIM / DT)
W2      = weights_tier2()

def run_sim(get_quotes_1, get_quotes_2,
            Lambda0_1, Lambda0_2,
            alpha2=0.0, beta2=0.0,
            epsilon=1.0, seed=0):
    # Returns array of final PnL for each simulation path.
    rng = np.random.default_rng(seed)
    q   = np.zeros(N_SIMS)
    pnl = np.zeros(N_SIMS)
    Lambda0_tot = Lambda0_1 + Lambda0_2

    for _ in range(N_STEPS):
        db1, da1 = get_quotes_1(q, Lambda0_1)
        db2, da2 = get_quotes_2(q, Lambda0_2)

        # Brownian PnL: q * sigma * dB
        dB    = rng.standard_normal(N_SIMS) * np.sqrt(DT)
        pnl  += q * SIGMA * dB

        # Price reading drift contribution to PnL (via q * dS_PR term)
        # dS_PR = epsilon * J^n(sum_k w^{n,k}*(delta_a - delta_b)) * dt
        # PnL contribution = q * dS_PR
        skew2 = np.sum(W2[:,None] * (da2 - db2), axis=0)   # (N_SIMS,)
        pnl  += q * epsilon * skew2 * DT

        # Adverse selection parameters per tier
        params = [
            (db1, da1, Lambda0_1, 0.0,   0.0),    # tier 1: no AS
            (db2, da2, Lambda0_2, alpha2, beta2),  # tier 2: with AS
        ]
        for db_n, da_n, Lam_n, a, b in params:
            for k in range(K):
                dk = SIZES[k]
                # --- Bid side (client sells to MM) ---------------------------
                lam_b = Lam_n[k] * np.exp(-KAPPA * db_n[k])   # (N_SIMS,)
                nb    = rng.poisson(lam_b * DT)                # (N_SIMS,)
                if nb.any():
                    zeta_b      = a * np.exp(b * db_n[k])
                    as_cost_b   = (q + dk) / dk * epsilon * zeta_b
                    pnl        += dk * (db_n[k] - as_cost_b) * nb
                    q          += dk * nb

                # --- Ask side (client buys from MM) --------------------------
                lam_a = Lam_n[k] * np.exp(-KAPPA * da_n[k])
                na    = rng.poisson(lam_a * DT)
                if na.any():
                    zeta_a      = a * np.exp(b * da_n[k])
                    as_gain_a   = (q - dk) / dk * epsilon * zeta_a
                    pnl        += dk * (da_n[k] + as_gain_a) * na
                    q          -= dk * na

        # Inventory penalty: -0.5 * gamma * sigma^2 * q^2 * dt
        pnl -= 0.5 * GAMMA * SIGMA**2 * q**2 * DT

    return pnl

print(f"Simulation engine ready: {N_SIMS} sims x {N_STEPS} steps x {DT}s = {T_SIM:.0f}s total.")


## 6. Price Reading PnL Simulation (Figs 3-5)
Three strategies (Sec. 5.2): No Action, No Skew, Optimal.

In [ ]:
# --- Strategy factories for price reading ------------------------------------

def make_no_action(Lambda0_tot):
    # Baseline quotes shown to all tiers regardless of sniffers.
    def get_quotes(q, L_n):
        return baseline_quotes(q, Lambda0_tot)
    return get_quotes

def make_no_skew(Lambda0_tot):
    # Tier 1: baseline; Tier 2: flat quotes (no skew).
    def get_quotes_t1(q, L_n):
        return baseline_quotes(q, Lambda0_tot)
    def get_quotes_t2(q, L_n):
        q_ = np.asarray(q)
        flat = np.full((K, N_SIMS), 1.0/KAPPA)
        return flat, flat
    return get_quotes_t1, get_quotes_t2

def make_optimal_pr(Lambda0_1, Lambda0_2):
    Lambda0_tot = Lambda0_1 + Lambda0_2
    w2 = weights_tier2()
    sum_w = np.sum(w2)
    denom = KAPPA * np.sum(SIZES * Lambda0_tot)
    global_coeff = E * sum_w / denom

    def get_quotes_tier(n_tier, Lambda0_tot):
        w_n = w2 if n_tier == 1 else np.zeros(K)
        def get_quotes(q, L_n):
            db_base, da_base = baseline_quotes(q, Lambda0_tot)
            q_ = np.asarray(q)
            # Global component
            gb = global_coeff * (q_[None,:] + SIZES[:,None]/2.0)
            ga = global_coeff * (-q_[None,:] + SIZES[:,None]/2.0)
            # Tier-specific component
            Lam_b = Lambda0_tot[:,None] * np.exp(-KAPPA * db_base)
            Lam_a = Lambda0_tot[:,None] * np.exp(-KAPPA * da_base)
            tb = -(q_[None,:] * w_n[:,None]) / (SIZES[:,None] * KAPPA * Lam_b)
            ta =  (q_[None,:] * w_n[:,None]) / (SIZES[:,None] * KAPPA * Lam_a)
            db = np.maximum(db_base + gb + tb, 0.0)
            da = np.maximum(da_base + ga + ta, 0.0)
            return db, da
        return get_quotes
    return get_quotes_tier(0, Lambda0_tot), get_quotes_tier(1, Lambda0_tot)


# --- Sweep SVS (Sec. 5.2) ----------------------------------------------------
SVS_GRID = np.array([0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90])
pr_results = {k: [] for k in ['no_reading', 'no_action', 'no_skew', 'optimal']}

print("Running price reading simulations (may take a few minutes)...")
for i, svs in enumerate(SVS_GRID):
    L1 = svs * LAMBDA0_TOTAL
    L2 = (1-svs) * LAMBDA0_TOTAL
    Lt = LAMBDA0_TOTAL

    # No Reading baseline (w2=0 effectively: no skew sniffer response)
    na_fn = make_no_action(Lt)
    pnl_nr = run_sim(na_fn, na_fn, Lt, np.zeros(K), 0., 0., seed=100+i)

    # No Action
    na1 = make_no_action(L1+L2)
    pnl_na = run_sim(na1, na1, L1, L2, 0., 0., seed=200+i)

    # No Skew
    ns1, ns2 = make_no_skew(L1+L2)
    pnl_ns = run_sim(ns1, ns2, L1, L2, 0., 0., seed=300+i)

    # Optimal
    op1, op2 = make_optimal_pr(L1, L2)
    pnl_op = run_sim(op1, op2, L1, L2, 0., 0., seed=400+i)

    for key, pnl in zip(pr_results, [pnl_nr, pnl_na, pnl_ns, pnl_op]):
        pr_results[key].append((pnl.mean(), pnl.std()))

    print(f"  SVS={svs:.0%}: no_action={pnl_na.mean():+.1f}  optimal={pnl_op.mean():+.1f}  (bp*M)")

print("Done.")


In [ ]:
# --- Figures 3, 4, 5 (Sec. 5.2) ---------------------------------------------
styles  = {'no_reading':('green',':','o'), 'no_action':('red','--','s'),
           'no_skew':('gray','-','o'),     'optimal':('black','-','o')}
labels  = {'no_reading':'No Reading', 'no_action':'No Action',
           'no_skew':'No Skew',        'optimal':'Optimal'}

def ex(key):
    mus  = np.array([v[0] for v in pr_results[key]])
    stds = np.array([v[1] for v in pr_results[key]])
    return mus, stds

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xpct = SVS_GRID * 100

for ax, metric, title, ylabel in zip(
        axes,
        ['mean', 'std', 'ratio'],
        ['Figure 3 - Average PnL (PR)',
         'Figure 4 - PnL Risk (PR)',
         'Figure 5 - Risk-Adjusted Performance (PR)'],
        ['PnL (bp*M)', 'Risk sigma(PnL) (bp*M)', 'PnL / Risk']):
    for key in pr_results:
        mus, stds = ex(key)
        c, ls, mk = styles[key]
        vals = mus if metric == 'mean' else (stds if metric == 'std' else mus/stds)
        ax.plot(xpct, vals, color=c, ls=ls, marker=mk, ms=5, label=labels[key])
    ax.set(xlabel='Safe Volume Share (%)', ylabel=ylabel, title=title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig('/tmp/figs345.png', dpi=110); plt.show()


**Figs 3-5 interpretation**:
- **No Action** reduces PnL vs baseline; risk unchanged
- **No Skew** partially restores PnL but greatly increases risk (especially low SVS)
- **Optimal** improves both PnL and risk; most effective when sniffers are rare (high SVS)

[WARNING] **Exhibit Note**: Units are bp*M not k$ (paper uses specific FX notional scaling).
Qualitative ranking of strategies matches the paper exactly.


## 7. Adverse Selection: Quote Adjustments (Fig 6)

**Setup** (Sec. 5.3): $\alpha^{1,k}=0$; $\alpha^{2,k}=0.05$; slow signals $\beta^{2,k}=2<\kappa=3$.

**Closed-form bid adjustment** (Sec. 5.3):
$$\frac{\hat{d}^{n,k,b*}(q)-\hat{\delta}^{n,k,b*}(q)}{\varepsilon}
= \frac{\sum_{m,j}\alpha^{m,j}\Lambda_0^{m,j}(\beta^{m,j}-\kappa)e^{\beta^{m,j}/\kappa}}{\sum_{m,j}\Delta^j\Lambda_0^{m,j}\kappa}\left(q+\frac{\Delta^k}{2}\right)
- \frac{q+\Delta^k}{\Delta^k}\cdot\frac{\beta^{n,k}-\kappa}{\kappa}\cdot\zeta^{n,k}(\hat{\delta}^{n,k,b*}(q))$$


In [ ]:
# --- Adverse selection quote adjustment (Sec. 5.3) --------------------------
ALPHA2 = 0.05
BETA2  = 2.0     # slow signal: beta < kappa

def as_adj_bid(q_arr, n_tier, Lambda0_1, Lambda0_2, alpha2, beta2):
    Lambda0_tot = Lambda0_1 + Lambda0_2
    # Global component numerator: sum alpha*Lambda_0*(beta-kappa)*exp(beta/kappa)
    # Only tier 2 contributes (alpha1=0)
    num_global = np.sum(alpha2 * Lambda0_2 * (beta2 - KAPPA) * np.exp(beta2/KAPPA))
    denom      = KAPPA * np.sum(SIZES * Lambda0_tot)
    global_c   = num_global / denom

    q = np.asarray(q_arr)
    db_base, _ = baseline_quotes(q, Lambda0_tot)  # (K, Q)

    # Global term (bid)
    global_bid = global_c * (q[None,:] + SIZES[:,None]/2.0)  # (K, Q)

    # Tier-specific term: -(q+Delta^k)/Delta^k * (beta-kappa)/kappa * zeta(delta_base)
    if n_tier == 1:
        a_n, b_n = alpha2, beta2
    else:
        a_n, b_n = 0.0, 0.0

    zeta_b = a_n * np.exp(b_n * db_base)     # (K, Q)
    tier_bid = -((q[None,:] + SIZES[:,None]) / SIZES[:,None]) * ((b_n - KAPPA) / KAPPA) * zeta_b

    return global_bid + tier_bid

# --- Figure 6: SVS=50%, slow AS signals (Sec. 5.3) --------------------------
SVS = 0.50
L1, L2 = SVS * LAMBDA0_TOTAL, (1-SVS) * LAMBDA0_TOTAL

adj_t1 = as_adj_bid(q_grid, 0, L1, L2, ALPHA2, BETA2)
adj_t2 = as_adj_bid(q_grid, 1, L1, L2, ALPHA2, BETA2)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(q_grid, adj_t1[0],  'k--',  lw=1.5, label='tier 1, 1M')
ax.plot(q_grid, adj_t1[3],  'k-',   lw=1.5, label='tier 1, 10M')
ax.plot(q_grid, adj_t2[0],  color='gray', lw=1.5, ls='--', label='tier 2, 1M')
ax.plot(q_grid, adj_t2[3],  color='gray', lw=1.5, ls='-',  label='tier 2, 10M')
ax.axhline(0, color='k', lw=0.5, ls=':'); ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set(xlim=(-60,60), ylim=(-0.5,0.5), xlabel='Inventory (M)', ylabel='Bid quote adjustment',
       title='Figure 6 - Adverse selection bid adj, SVS=50%, beta2=2 < kappa=3')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('/tmp/fig6.png', dpi=110); plt.show()


**Figure 6 interpretation** (slow signals, $\beta < \kappa$):
- **Tier 1** (uninformed): spread *tightened* at q=0 — MM acts less risk-averse, using informed trades as a signal subscription
- **Tier 2** (informed): spread *widened* — compensation for adverse selection losses
- Small risk-reducing trades with informed clients provide directional information at modest cost (Sec. 5.3)


## 8. Adverse Selection PnL Simulation (Figs 7-9)
Two strategies (Sec. 5.3): No Action vs Optimal.

In [ ]:
# --- Optimal AS strategy (Eqs. 12-13) ----------------------------------------

def make_optimal_as(Lambda0_1, Lambda0_2, alpha2, beta2):
    Lambda0_tot = Lambda0_1 + Lambda0_2
    num_global = np.sum(alpha2 * Lambda0_2 * (beta2 - KAPPA) * np.exp(beta2/KAPPA))
    denom      = KAPPA * np.sum(SIZES * Lambda0_tot)
    global_c   = num_global / denom

    def make_tier_fn(n_tier):
        a_n = alpha2 if n_tier == 1 else 0.0
        b_n = beta2  if n_tier == 1 else 0.0
        def get_quotes(q, L_n):
            db_base, da_base = baseline_quotes(q, Lambda0_tot)
            q_ = np.asarray(q)
            # Global component
            gb = global_c * (q_[None,:] + SIZES[:,None]/2.0)
            ga = global_c * (-q_[None,:] + SIZES[:,None]/2.0)
            # Tier-specific component (bid: -(q+Dk)/Dk, ask: +(q-Dk)/Dk)
            zeta_b = a_n * np.exp(b_n * db_base)
            zeta_a = a_n * np.exp(b_n * da_base)
            tb = -((q_[None,:] + SIZES[:,None]) / SIZES[:,None]) * ((b_n - KAPPA)/KAPPA) * zeta_b
            ta =  ((q_[None,:] - SIZES[:,None]) / SIZES[:,None]) * ((b_n - KAPPA)/KAPPA) * zeta_a
            db = np.maximum(db_base + gb + tb, 0.0)
            da = np.maximum(da_base + ga + ta, 0.0)
            return db, da
        return get_quotes
    return make_tier_fn(0), make_tier_fn(1)


# --- Sweep SVS (informed fraction = 1 - SVS) ---------------------------------
# Paper's x-axis: 0.10% to 0.90% informed volume
SVS_AS = np.linspace(0.001, 0.009, 9)    # 0.1% to 0.9% uninformed fraction
as_results = {k: [] for k in ['no_adverse', 'no_action', 'optimal']}

print("Running adverse selection simulations...")
for i, svs in enumerate(SVS_AS):
    L1 = svs * LAMBDA0_TOTAL
    L2 = (1-svs) * LAMBDA0_TOTAL
    na_fn = make_no_action(L1+L2)

    # No Adverse Selection baseline (alpha2=0, same intensities)
    pnl_nb = run_sim(na_fn, na_fn, L1, L2, 0.0, 0.0, seed=500+i)

    # No Action (alpha2 active, no quote adjustment)
    pnl_na = run_sim(na_fn, na_fn, L1, L2, ALPHA2, BETA2, seed=600+i)

    # Optimal
    op1, op2 = make_optimal_as(L1, L2, ALPHA2, BETA2)
    pnl_op = run_sim(op1, op2, L1, L2, ALPHA2, BETA2, seed=700+i)

    for key, pnl in zip(as_results, [pnl_nb, pnl_na, pnl_op]):
        as_results[key].append((pnl.mean(), pnl.std()))

    informed_pct = (1-svs)*100
    print(f"  Informed={informed_pct:.1f}%: no_action={pnl_na.mean():+.1f}  optimal={pnl_op.mean():+.1f}")

print("Done.")


In [ ]:
# --- Figures 7, 8, 9 (Sec. 5.3) ---------------------------------------------
styles_as = {'no_adverse':('green',':','o'), 'no_action':('red','--','s'), 'optimal':('black','-','o')}
labels_as = {'no_adverse':'No Adverse Selection', 'no_action':'No Action', 'optimal':'Optimal'}

def ex_as(key):
    mus  = np.array([v[0] for v in as_results[key]])
    stds = np.array([v[1] for v in as_results[key]])
    return mus, stds

x_inf = (1 - SVS_AS) * 100   # informed volume %

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, title, ylabel in zip(
        axes,
        ['mean', 'std', 'ratio'],
        ['Figure 7 - Average PnL (AS)',
         'Figure 8 - PnL Risk (AS)',
         'Figure 9 - Risk-Adjusted Performance (AS)'],
        ['PnL (bp*M)', 'Risk sigma(PnL) (bp*M)', 'PnL / Risk']):
    for key in as_results:
        mus, stds = ex_as(key)
        c, ls, mk = styles_as[key]
        vals = mus if metric == 'mean' else (stds if metric == 'std' else mus/stds)
        ax.plot(x_inf, vals, color=c, ls=ls, marker=mk, ms=5, label=labels_as[key])
    ax.set(xlabel='Informed Volume Share (%)', ylabel=ylabel, title=title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig('/tmp/figs789.png', dpi=110); plt.show()


**Figs 7-9 interpretation**:
- **No Action** significantly reduces PnL vs the no-AS baseline
- **Optimal** protects PnL and substantially reduces risk, particularly for moderate informed share
- Risk-adjusted performance (Fig 9) improves consistently with no strong SVS dependence


## 9. Key Takeaways

| Risk Type | Global Effect on Value fn | Optimal Quote Response |
|---|---|---|
| **Price reading** | Increased concavity (higher risk aversion) | Widen spread at q=0; reduce skew to sniffers |
| **AS - slow signals** (beta < kappa) | Reduced concavity | Tighten safe-tier spread; widen informed-tier |
| **AS - fast signals** (beta > kappa) | Increased concavity | Wide spreads everywhere; defensive posture |

**Two-component structure** (Sec. 3.3):
1. **Global** ($D^k_\pm f$): applies uniformly across all tiers; encodes systemic informational risk
2. **Tier-specific** ($g^{n,k,b/a}$): fine-tunes per-client exposure; can push *opposite* to global

**Price reading trade-off**: global component encourages more skew (exit inventory faster);
tier-specific component pushes *against* skew (hide inventory from sniffers). Winner depends on
sniffers' trading intensity relative to risk-reduction contribution.

**Adverse selection as signal subscription**: a tier with slow-signal informed traders can
*benefit* the market maker by revealing price direction at modest execution cost (Sec. 5.3).


## Bibliography

1. Avellaneda, M., & Stoikov, S. (2008). *High-frequency trading in a limit order book*. Quantitative Finance, 8(3), 217-224.
2. Baldauf, M., & Mollner, J. (2024). *Competition and Information Leakage*. Journal of Political Economy, 132(5), 1603-1641.
3. Barzykin, A., Bergault, P., & Gueant, O. (2023). *Algorithmic market making in dealer markets with hedging and market impact*. Mathematical Finance, 33(1), 41-79.
4. Barzykin, A., Bergault, P., & Gueant, O. (2023). *Dealing with multi-currency inventory risk in FX cash markets*. Risk, March 2023.
5. Barzykin, A., Bergault, P., & Gueant, O. (2024). *Market-making in spot precious metals*. Risk, December 2024.
6. Bergault, P., Evangelista, D., Gueant, O., & Vieira, D. (2021). *Closed-form approximations in multi-asset market making*. Applied Mathematical Finance, 28(2), 101-142.
7. Butz, M., & Oomen, R. (2019). *Internalisation by electronic FX spot dealers*. Quantitative Finance, 19(1), 35-56.
8. Cartea, A., Jaimungal, S., & Penalva, J. (2015). *Algorithmic and High-Frequency Trading*. Cambridge University Press.
9. Cartea, A., Jaimungal, S., & Ricci, J. (2014). *Buy low, sell high: A high frequency trading perspective*. SIAM Journal on Financial Mathematics, 5(1), 415-444.
10. Cartea, A., & Sanchez-Betancourt, L. (2025). *A Simple Strategy to Deal with Toxic Flow*. arXiv:2503.18005.
11. Glosten, L.R., & Milgrom, P.R. (1985). *Bid, ask and transaction prices in a specialist market*. Journal of Financial Economics, 14(1), 71-100.
12. Goyder, B., & Lipsky, C. (2024). *BNPP ups efforts to weed out skew sniffers*. FX Markets.
13. Gueant, O. (2016). *The Financial Mathematics of Market Liquidity*. CRC Press.
14. Gueant, O., Lehalle, C.-A., & Fernandez-Tapia, J. (2013). *Dealing with the inventory risk*. Mathematics and Financial Economics, 7(4), 477-507.
15. Ho, T., & Stoll, H.R. (1981). *Optimal dealer pricing under transactions and return uncertainty*. Journal of Financial Economics, 9(1), 47-73.
16. Lucchese, L., Pakkanen, M.S., & Veraart, A.E.D. (2024). *Short-term predictability of returns in order book markets*. International Journal of Forecasting, 40(4), 1587-1621.
17. Rosenbaum, M., & Zhang, J. (2022). *Multi-asset market making under the quadratic rough Heston*. arXiv:2212.10164.
18. Sirignano, J., & Cont, R. (2019). *Universal features of price formation in financial markets*. Quantitative Finance, 19(9), 1449-1459.
